# ✍ Conditional VAE — MNIST Digit Generator
### Train from scratch → Deploy as public Gradio app

**Before running:** `Runtime → Change runtime type → T4 GPU`

| Step | Cell | Time |
|------|------|------|
| 1. Check GPU | Cell 1 | ~5s |
| 2. Install deps | Cell 2 | ~1 min |
| 3. Write scripts | Cell 3 | instant |
| 4. Train model | Cell 4 | ~20-25 min |
| 5. Preview samples | Cell 5 | ~5s |
| 6. Launch Gradio | Cell 6 | ~10s |
| 7. Deploy to HF | Cell 7 | ~2 min |

In [ ]:
# ═══════════════════════════════════════════
# CELL 1 — GPU Check
# ═══════════════════════════════════════════
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name        : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM            : {vram:.1f} GB')
    if vram < 12:
        print('WARNING: < 12GB VRAM detected. Reduce batch_size to 128 if OOM.')
else:
    print('No GPU! Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# ═══════════════════════════════════════════
# CELL 2 — Install / verify dependencies
# ═══════════════════════════════════════════
# torchvision already present in Colab; gradio needed for app only
!pip install -q gradio==4.44.0 huggingface_hub
print('Done.')

In [ ]:
# ═══════════════════════════════════════════
# CELL 3 — Write training + app scripts
# ═══════════════════════════════════════════
# Option A: download from your GitHub/Gist
# !wget -q https://raw.githubusercontent.com/YOUR/REPO/main/train_cvae_mnist.py
# !wget -q https://raw.githubusercontent.com/YOUR/REPO/main/app.py

# Option B: use the files you uploaded to Colab
import os
for f in ['train_cvae_mnist.py', 'app.py', 'requirements.txt']:
    if os.path.exists(f):
        print(f'  found: {f}')
    else:
        print(f'  MISSING: {f}  <- upload this file to Colab first')

In [ ]:
# ═══════════════════════════════════════════
# CELL 4 — Train the CVAE
# ═══════════════════════════════════════════
# Full training: 60 epochs ~20-25 min on T4
# Quick smoke test: --epochs 3

!python train_cvae_mnist.py \
    --epochs 60 \
    --batch_size 256 \
    --latent_dim 128 \
    --lr 2e-4 \
    --beta 4.0

# Expected output at epoch 60:
#   recon ~ 60-80   kl ~ 8-15   (lower = better reconstruction)

In [ ]:
# ═══════════════════════════════════════════
# CELL 5 — Preview generated samples
# ═══════════════════════════════════════════
from IPython.display import Image as IPImage, display
import os, glob

# Show final epoch grid
grids = sorted(glob.glob('samples/epoch_*.png'))
if grids:
    print(f'Final grid ({grids[-1]}):')
    display(IPImage(grids[-1], width=700))

# Show temperature comparison
for tag in ['sharp', 'balanced', 'diverse']:
    p = f'samples/final_{tag}.png'
    if os.path.exists(p):
        print(f'\nTemperature: {tag}')
        display(IPImage(p, width=700))

# Show per-digit test strips
for d in range(10):
    p = f'samples/test_digit_{d}.png'
    if os.path.exists(p):
        print(f'Digit {d}:', end=' ')
        display(IPImage(p, width=400))

In [ ]:
# ═══════════════════════════════════════════
# CELL 6 — Launch Gradio (public link)
# ═══════════════════════════════════════════
# This starts a local server + creates a public gradio.live tunnel
# The link (https://xxxxxx.gradio.live) is valid for 72 hours

import subprocess, sys

# Launch app.py in background so the cell doesn't block
proc = subprocess.Popen(
    [sys.executable, 'app.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

import time
print('Starting Gradio app ...')
for _ in range(30):
    line = proc.stdout.readline()
    if line:
        print(line.rstrip())
    if 'gradio.live' in line or 'Running on' in line:
        break
    time.sleep(1)

print('\nGradio server running. Copy the gradio.live link above!')

In [ ]:
# ═══════════════════════════════════════════
# CELL 7 — Deploy to HuggingFace Spaces
# ═══════════════════════════════════════════
# Prerequisites:
#   1. Account at huggingface.co
#   2. New Space: https://huggingface.co/new-space
#      Name: digit-generator | SDK: Gradio | Hardware: CPU Basic (free)
#   3. Token: https://huggingface.co/settings/tokens (write access)

HF_TOKEN = 'hf_YOUR_TOKEN_HERE'           # <-- replace
HF_REPO  = 'YOUR_USERNAME/digit-generator' # <-- replace

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)

files = ['app.py', 'cvae_mnist.pth', 'requirements.txt']
for fname in files:
    if not os.path.exists(fname):
        print(f'SKIP (not found): {fname}')
        continue
    api.upload_file(
        path_or_fileobj=fname,
        path_in_repo=fname,
        repo_id=HF_REPO,
        repo_type='space',
        token=HF_TOKEN,
    )
    print(f'Uploaded: {fname}')

print(f'\nApp live at: https://huggingface.co/spaces/{HF_REPO}')

In [ ]:
# ═══════════════════════════════════════════
# CELL 8 (optional) — Save model to Google Drive
# ═══════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import shutil
dest = '/content/drive/MyDrive/cvae_mnist'
os.makedirs(dest, exist_ok=True)
shutil.copy('cvae_mnist.pth', f'{dest}/cvae_mnist.pth')
shutil.copytree('samples', f'{dest}/samples', dirs_exist_ok=True)
print(f'Saved to Google Drive: {dest}')